# Quantifier l'héritage de conception partagé dans un parc de transformateurs de puissance avec PROC INBREED

## Résumé analytique

Les transformateurs de poste d'un opérateur de réseau sont conçus en générations successives, chaque nouveau modèle étant dérivé de deux conceptions prédécesseurs. Ce notebook traite cette généalogie d'ingénierie comme un pedigree et utilise **PROC INBREED** pour calculer les coefficients de consanguinité et de parenté additive (covariance), quantifiant combien d'héritage de conception deux actifs quelconques partagent.

Le pedigree est construit de sorte que la structure soit interprétable : deux des quatre modèles du parc actuel descendent d'une paire de conceptions prédécesseurs **germaines** et portent donc un héritage concentré, tandis que les autres puisent dans des lignées disjointes. La procédure retrouve cela exactement. Les deux modèles issus de germains (`G3_T1`, `G3_T3`) portent chacun un coefficient de consanguinité de **F = 0,25** ; les deux autres (`G3_T2`, `G3_T4`) sont à **F = 0**. Le coefficient de consanguinité moyen du parc est de **0,0417**. En lisant la matrice de parenté additive, la paire du parc actuel la moins apparentée est **`G3_T1` et `G3_T3` (parenté = 0)** — elles ne partagent aucune ascendance et forment le meilleur appariement redondant (N-1), ce qui importe car `G3_T1` est lui-même l'une des conceptions les plus consanguines.

## Sources de données

| Jeu de données | Lignes | Variables clés | Description |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | Un pedigree d'ingénierie petit et déterministe de conceptions de transformateurs de poste, réparti sur trois générations de conception non chevauchantes (4 plateformes fondatrices, 4 dérivés de deuxième génération, 4 modèles du parc actuel). Chaque conception non fondatrice enregistre les deux conceptions prédécesseurs distinctes (`Parent1`, `Parent2`) dont elle est dérivée. `Sex` marque le rôle de conception principal (M = lignée mécanique/noyau, F = lignée électrique/bobinage). Le pedigree est spécifié à la main — pas aléatoire — de sorte que la structure de parenté est connue à l'avance et peut être vérifiée par rapport à la sortie de la procédure. |

# Quantifier l'héritage de conception partagé dans un parc de transformateurs de puissance

## Pourquoi un opérateur se soucie de la « consanguinité »

Un opérateur de transport et de distribution exploite des centaines de transformateurs de puissance de poste. Pour maîtriser le coût d'ingénierie et l'effort de qualification, les fabricants conçoivent rarement chaque transformateur à partir de zéro — chaque nouveau modèle **hérite** de la géométrie de base, de la topologie du bobinage, des systèmes d'isolation et des conceptions de traversées d'un ou deux modèles prédécesseurs. Sur plusieurs cycles d'approvisionnement, cela produit une véritable *généalogie d'ingénierie* : un pedigree de conceptions.

Cet héritage partagé est un risque caché pour la fiabilité. Si deux transformateurs qui protègent la même charge (une paire redondante **N-1**) descendent de la même conception ancestrale, un défaut de conception latent — un mode de résonance, un mécanisme de vieillissement de l'isolation, une voie de contournement de traversée — est susceptible d'être présent dans **les deux** unités. Une cause unique peut alors mettre hors service la paire redondante simultanément : une *défaillance de mode commun*.

**PROC INBREED** a été conçu pour quantifier exactement ce type d'ascendance partagée. Bien que ses origines soient dans l'élevage animal et végétal, ses mathématiques — la récursion de parenté additive de Wright — sont indépendantes du domaine : elle mesure la fraction d'héritage de conception que deux individus partagent via des ancêtres communs. Nous transposons le vocabulaire génétique vers l'ingénierie des actifs :

| Concept INBREED | Interprétation pour l'opérateur |
|---|---|
| Individu | Une conception / un modèle de transformateur |
| Parent (père/mère) | Une conception prédécesseure dont il est dérivé |
| Génération (CLASS) | Un cycle d'approvisionnement / de conception |
| Coefficient de consanguinité *F* | Degré d'héritage auto-similaire au sein d'une conception |
| Coefficient de covariance / parenté | Héritage de conception partagé entre deux conceptions |
| Paire la moins apparentée | Meilleur appariement redondant pour la résilience N-1 |

## Étape 1 — Spécifier le pedigree de conception

Nous saisissons `DesignLineage` directement afin que la structure de parenté soit sans ambiguïté. Il y a trois **générations de conception non chevauchantes** :

- **Génération 1** — quatre conceptions de plateforme fondatrices (`G1_P1`-`G1_P4`) sans prédécesseurs (parents vides).
- **Génération 2** — quatre conceptions dérivées (`G2_D1`-`G2_D4`), chacune conçue à partir de deux plateformes **distinctes** de génération 1. `G2_D1` et `G2_D2` sont *germaines* (toutes deux issues de `G1_P1` x `G1_P2`) ; `G2_D3` et `G2_D4` sont germaines (toutes deux issues de `G1_P3` x `G1_P4`).
- **Génération 3** — quatre modèles du parc actuel (`G3_T1`-`G3_T4`). `G3_T1` est construit à partir de la paire germaine `G2_D1` x `G2_D2`, et `G3_T3` à partir de la paire germaine `G2_D3` x `G2_D4` ; ces deux conceptions concentrent donc l'héritage. `G3_T2` et `G3_T4` croisent chacun les deux lignées disjointes.

Les colonnes `ID`, `Parent1` et `Parent2` portent le pedigree ; `Sex` enregistre quelle lignée d'ingénierie a mené la conception. Une courte étape DATA de suivi vide le symbole `.` afin que les plateformes fondatrices aient des parents vides, comme l'attend PROC INBREED.

In [1]:
DONNÉES DesignLineage;
   LONGUEUR ID $12 Parent1 $12 Parent2 $12 Sex $1;
   FICHIER_ENTRÉE CARTES truncover;
   ENTRÉE Generation ID $ Parent1 $ Parent2 $ Sex $;
   CARTES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
EXÉCUTER;

/* Les plateformes fondatrices n'ont pas de prédécesseurs : vider les points de remplissage */
DONNÉES DesignLineage;
   DÉFINIR DesignLineage;
   SI Parent1 = '.' ALORS Parent1 = ' ';
   SI Parent2 = '.' ALORS Parent2 = ' ';
EXÉCUTER;

TITRE 'Pedigree de conception des transformateurs';
PROCÉDURE IMPRIMER DONNÉES=DesignLineage noobs;
   VAR Generation ID Parent1 Parent2 Sex;
   ÉTIQUETTE Generation='Génération' ID='Identifiant' Parent1='Parent 1' Parent2='Parent 2' Sex='Rôle de conception';
EXÉCUTER;


                                       Pedigree de conception des transformateurs                                       


  Génération  Identifiant  Parent 1  Parent 2   Rôle de conception
------------  -----------  --------  --------  -------------------
           1  G1_P1                            M
           1  G1_P2                            M
           1  G1_P3                            M
           1  G1_P4                            F
           2  G2_D1        G1_P1     G1_P2     M
           2  G2_D2        G1_P1     G1_P2     F
           2  G2_D3        G1_P3     G1_P4     F
           2  G2_D4        G1_P3     G1_P4     M
           3  G3_T1        G2_D1     G2_D2     M
           3  G3_T2        G2_D1     G2_D3     F
           3  G3_T3        G2_D3     G2_D4     F
           3  G3_T4        G2_D2     G2_D4     M




NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to Pedigree de conception des transformateurs.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## Étape 2 — Coefficients de consanguinité par génération de conception

Nous exécutons PROC INBREED en **mode multi-génération** en nommant `Generation` dans l'instruction `CLASS`, ce qui imprime le résumé du pedigree par génération. L'instruction `VAR` fournit les trois colonnes du pedigree dans l'ordre requis : individu, premier prédécesseur, second prédécesseur.

- L'option **IND** imprime le coefficient de consanguinité *F* de chaque conception — une mesure de la concentration de son propre héritage. Une conception construite à partir de deux prédécesseurs étroitement apparentés porte un *F* élevé.
- L'option **MATRIX** imprime la matrice complète de parenté additive afin que nous puissions lire directement l'héritage par paire.
- L'option **AVERAGE** ajoute le coefficient de consanguinité moyen à l'échelle du parc — un résumé unique de la diversité des conceptions.

Des valeurs proches de 0 signifient des lignées indépendantes ; *F* augmente à mesure que les deux prédécesseurs d'une conception deviennent plus étroitement apparentés.

In [2]:
TITRE 'Coefficients de consanguinité par génération de conception';
PROCÉDURE inbreed DONNÉES=DesignLineage ind average MATRIX;
   VAR ID Parent1 Parent2;
   CLASSE Generation;
EXÉCUTER;


                               Coefficients de consanguinité par génération de conception                               


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Generation 2)
--------------------------------------
ID                  F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coe


NOTE: Option TITLE changed to Coefficients de consanguinité par génération de conception.
NOTE: PROC INBREED data=DesignLineage



## Étape 3 — Coefficients de covariance enregistrés pour la notation de risque en aval

En remplaçant la vue de consanguinité par l'option **COVAR**, on rapporte les mêmes relations sous forme de **coefficients de covariance (parenté additive)**, la forme qu'une équipe de gestion d'actifs injecterait dans un score de risque de redondance. L'option **OUTCOV=** capture la matrice complète sous forme de jeu de données (`DesignCovar`), où les colonnes `Col1`-`Col12` portent la parenté de chaque conception avec toutes les autres (dans l'ordre du pedigree) — prêtes à être jointes aux affectations de poste.

Nous imprimons le jeu de données en sortie afin que la matrice enregistrée soit visible à côté du listing.

In [3]:
TITRE 'Coefficients de covariance (parenté)';
PROCÉDURE inbreed DONNÉES=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   VAR ID Parent1 Parent2;
   CLASSE Generation;
EXÉCUTER;

TITRE 'Matrice de parenté OUTCOV= enregistrée pour la notation de risque';
PROCÉDURE IMPRIMER DONNÉES=DesignCovar noobs;
EXÉCUTER;


                                          Coefficients de covariance (parenté)                                          


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Generation 1)
--------------------------------------------------
ID                     G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.00


NOTE: Option TITLE changed to Coefficients de covariance (parenté).
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to Matrice de parenté OUTCOV= enregistrée pour la notation de risque.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## Étape 4 — Appariements les moins apparentés pour les installations redondantes (N-1)

La matrice `DesignCovar` enregistrée est exactement ce dont une étude de redondance a besoin. La parenté de la conception *i* à la conception *j* se trouve à la ligne *i*, colonne `Col`*j* (les colonnes suivent l'ordre du pedigree, donc `Col9`-`Col12` correspondent aux quatre modèles du parc actuel `G3_T1`-`G3_T4`).

Nous lisons les quatre lignes du parc actuel dans `DesignCovar`, formons chaque paire non ordonnée du parc actuel, y attachons son coefficient de parenté, et trions par ordre croissant de parenté. L'objectif est de choisir les paires redondantes dont l'héritage partagé est **le plus faible** — celles-ci minimisent le risque qu'un défaut de conception hérité désactive les deux moitiés d'une position protégée N-1.

In [4]:
/* Extraire les quatre lignes du parc actuel ; Col9..Col12 sont les parentés
   avec G3_T1..G3_T4 (ordre des colonnes du pedigree). Construire chaque paire non ordonnée. */
DONNÉES Pairs;
   DÉFINIR DesignCovar;
   OÙ ID DANS ('G3_T1','G3_T2','G3_T3','G3_T4');
   LONGUEUR DesignA $12 DesignB $12;
   TABLEAU g3{4} Col9 Col10 Col11 Col12;
   TABLEAU nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   FAIRE j = 1 JUSQU_À 4;
      DesignB = nm{j};
      SI DesignA < DesignB ALORS FAIRE;
         Relationship = g3{j};
         SORTIE;
      FIN;
   FIN;
   GARDER DesignA DesignB Relationship;
EXÉCUTER;

PROCÉDURE TRIER DONNÉES=Pairs; PAR Relationship; EXÉCUTER;

TITRE 'Appariements redondants (N-1) candidats, du moins apparenté au plus apparenté';
PROCÉDURE IMPRIMER DONNÉES=Pairs noobs;
   VAR DesignA DesignB Relationship;
   ÉTIQUETTE DesignA='Conception A' DesignB='Conception B' Relationship='Parenté';
EXÉCUTER;
TITRE;


                     Appariements redondants (N-1) candidats, du moins apparenté au plus apparenté                      


Conception A  Conception B   Parenté
------------  ------------  --------
G3_T1         G3_T3                0
G3_T2         G3_T4             0.25
G3_T1         G3_T2            0.375
G3_T1         G3_T4            0.375
G3_T2         G3_T3            0.375
G3_T3         G3_T4            0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Appariements redondants (N-1) candidats, du moins apparenté au plus apparenté.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Interprétation des résultats

**Coefficients de consanguinité (étape 2).** Toutes les plateformes fondatrices de génération 1 et tous les dérivés de génération 2 affichent *F* = 0 — par construction, aucun n'a deux prédécesseurs apparentés. Deux modèles de génération 3 affichent *F* = 0,25 : `G3_T1` (construit à partir de la paire germaine `G2_D1` x `G2_D2`) et `G3_T3` (à partir de la paire germaine `G2_D3` x `G2_D4`). Leurs prédécesseurs remontent à la *même* paire de plateformes fondatrices, si bien que leur héritage est concentré. Du point de vue de la fiabilité, ce sont les conceptions les plus exposées à un défaut hérité unique, et elles justifient des essais de qualification indépendants supplémentaires. `G3_T2` et `G3_T4`, qui croisent les deux lignées disjointes, sont à *F* = 0.

**Matrice de parenté (étape 3).** Les entrées hors diagonale quantifient l'héritage partagé par paire. Les deux paires germaines de génération 2 affichent chacune une parenté de 0,50 l'une envers l'autre (`G2_D1`-`G2_D2` et `G2_D3`-`G2_D4`), tandis que les dérivés de lignées opposées se situent à 0,00. Les conceptions consanguines du parc actuel portent des parentés à soi-même de 1,25 (= 1 + *F*), visibles sur la diagonale. Le jeu de données `DesignCovar` rend la matrice complète disponible pour la jonction avec les affectations de poste.

**Appariements les moins apparentés (étape 4).** En classant chaque paire du parc actuel par parenté, **`G3_T1` et `G3_T3` arrivent en tête avec une parenté de 0,00** — elles descendent de plateformes ancestrales entièrement disjointes et ne partagent aucun héritage de conception. C'est l'appariement redondant le plus solide, et il est particulièrement précieux car `G3_T1` est lui-même l'une des deux conceptions les plus consanguines : l'apparier avec un partenaire totalement non apparenté couvre son héritage concentré. La paire suivante est `G3_T2` et `G3_T4` à 0,25 ; les paires restantes se situent à 0,375. Le coefficient de consanguinité moyen du parc de **0,0417** (imprimé par l'option AVERAGE à l'étape 2) résume la diversité globale des conceptions. Les achats devraient privilégier les paires les moins apparentées pour les positions N-1 et, avec le temps, élargir la base ancestrale — l'équivalent, en ingénierie des actifs, d'éviter un goulot d'étranglement génétique.

**Mise en garde.** Il s'agit de données synthétiques illustratives ; une étude de production construirait le pedigree à partir des véritables enregistrements de révision de conception du fabricant et validerait les scores de parenté par rapport à l'historique des incidents à mode commun avant de guider les décisions d'achat.